# Runtime palette and figure checks

This tutorial checks palettes that exist at runtime and colours extracted from a completed Matplotlib figure. For source-code and notebook scanning, use the [static linter](../static-linter); for command-line and pre-commit workflows, see [CLI, configuration, and CI](../cli-configuration).

In [13]:
from cvdlint import palette_check

## Check an explicit palette

`palette_check()` returns structured data instead of printing a terminal report.

In [14]:
palette = ["#E41A1C", "#4DAF4A", "#377EB8"]
result = palette_check(palette)
result.passed

False

In [15]:
for problem in result.problems:
    print(
        f"{problem.condition}: {problem.first_color} / "
        f"{problem.second_color} = {problem.distance:.2f} "
        f"< {problem.tolerance:.2f}"
    )

deuteranopia: #E41A1C / #4DAF4A = 9.60 < 10.00


Use `raise_for_failure()` in tests or pipelines. Here the exception is caught so the tutorial can continue.

In [16]:
try:
    result.raise_for_failure()
except AssertionError as error:
    print(error)

Palette is not CVD-safe:
  deuteranopia: #E41A1C / #4DAF4A = 9.60 < 10.00


## Check a passing palette

In [17]:
accessible = ["#440154", "#21918C", "#FDE725"]
palette_check(accessible).passed

True

## Adjust the policy

The Python API and CLI both default to an absolute CIEDE2000 tolerance of `10`. Tolerance, simulation severity, and metric can be changed explicitly.

In [18]:
stricter = palette_check(accessible, tolerance=15, severity=0.8)
stricter.passed

True

## Check a Matplotlib figure

The adapter inspects visible colours stored on the completed figure. This example uses Matplotlib's `Set1` colormap rather than passing a palette to cvdlint.

In [19]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from cvdlint.adapters.matplotlib import check_figure

In [20]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(
    ["A", "B", "C"],
    [3, 2, 4],
    color=plt.colormaps["Set1"].colors[:3],
)
ax.set(title="A Set1 bar chart", ylabel="Value")
fig

<Figure size 600x300 with 1 Axes>

In [21]:
figure_result = check_figure(fig)
for problem in figure_result.problems:
    print(
        f"{problem.condition}: {problem.first_color} / "
        f"{problem.second_color} = {problem.distance:.2f}"
    )

deuteranopia: #E41A1C / #4DAF4A = 9.60


Replace `Set1` with three colours sampled from `viridis`, then check the resulting figure again.

In [22]:
viridis = [plt.colormaps["viridis"](x) for x in (0.0, 0.5, 1.0)]
safe_fig, safe_ax = plt.subplots(figsize=(6, 3))
safe_ax.bar(["A", "B", "C"], [3, 2, 4], color=viridis)
safe_ax.set(title="A viridis bar chart", ylabel="Value")
check_figure(safe_fig).passed

True

## Check a Seaborn figure

Seaborn creates Matplotlib figures, so it uses the same adapter. Here Seaborn selects and renders the named `Set1` palette.

In [23]:
import seaborn as sns

seaborn_fig, seaborn_ax = plt.subplots(figsize=(6, 3))
sns.barplot(
    x=["A", "B", "C"],
    y=[3, 2, 4],
    hue=["A", "B", "C"],
    palette="Set1",
    ax=seaborn_ax,
)
seaborn_ax.set(title="A Seaborn Set1 bar chart", ylabel="Value")
seaborn_fig

<Figure size 600x300 with 1 Axes>

In [24]:
seaborn_result = check_figure(seaborn_fig)
for problem in seaborn_result.problems:
    print(
        f"{problem.condition}: {problem.first_color} / "
        f"{problem.second_color} = {problem.distance:.2f}"
    )

deuteranopia: #CB3335 / #59A257 = 9.50


## Check a Plotly figure

The Plotly adapter checks hexadecimal colours explicitly assigned to traces. It does not report unused colours expanded from a Plotly template.

In [25]:
import plotly.graph_objects as go
from cvdlint.adapters.plotly import check_figure as check_plotly_figure

plotly_fig = go.Figure(
    go.Bar(
        x=["A", "B", "C"],
        y=[3, 2, 4],
        marker_color=["#E41A1C", "#377EB8", "#4DAF4A"],
    )
)
_ = plotly_fig.update_layout(title="A Plotly Set1 bar chart")

In [26]:
plotly_result = check_plotly_figure(plotly_fig)
for problem in plotly_result.problems:
    print(
        f"{problem.condition}: {problem.first_color} / "
        f"{problem.second_color} = {problem.distance:.2f}"
    )

deuteranopia: #E41A1C / #4DAF4A = 9.60


## Check a PyROOT canvas

PyROOT is distributed by the ROOT project rather than installed by a cvdlint extra. The following cell is included as a runnable example for a ROOT environment and is skipped by the documentation build.

In [27]:
import uuid

import ROOT
from cvdlint.adapters.root import check_canvas

ROOT.gStyle.SetOptStat(0)
run_id = uuid.uuid4().hex
canvas = ROOT.TCanvas(f"cvdlint_canvas_{run_id}", "Set1 bars", 600, 400)
categories = ["A", "B", "C"]
values = [3, 2, 4]
colours = ["#E41A1C", "#377EB8", "#4DAF4A"]
histograms = []
for index, (value, colour) in enumerate(zip(values, colours)):
    histogram = ROOT.TH1F(
        f"cvdlint_h{index}_{run_id}",
        "Set1 bar chart;Category;Value",
        3,
        0,
        3,
    )
    for bin_index, category in enumerate(categories, start=1):
        histogram.GetXaxis().SetBinLabel(bin_index, category)
    histogram.SetBinContent(index + 1, value)
    root_colour = ROOT.TColor.GetColor(colour)
    histogram.SetFillColor(root_colour)
    histogram.SetLineColor(root_colour)
    histogram.SetBarWidth(0.8)
    histogram.SetBarOffset(0.1)
    histogram.SetMinimum(0)
    histogram.SetMaximum(5)
    histogram.Draw("BAR SAME" if index else "BAR")
    histograms.append(histogram)
canvas.Update()
root_result = check_canvas(canvas)
if root_result.problems:
    print("ERROR")
for problem in root_result.problems:
    print(
        f"{problem.condition}: {problem.first_color} / "
        f"{problem.second_color} = {problem.distance:.2f}"
    )
canvas

ERROR
deuteranopia: #E41A1C / #4DAF4A = 9.60
